# Простая модель прогноза дохода с ориентиром около 76k

Здесь строятся две связанные модели на основе **LightGBM**. 

1. **Основная модель** оценивает доход по 396 признакам.
2. **Модель-поправка** смотрит, в каких случаях основная модель обычно ошибается, и
   осторожно меняет её прогноз. Для поправки используются 73 понятных признака.


Проверка на последних доступных месяцах дала:

- May: `59 606 → 58 680`;
- June: `59 312 → 58 761`;
- pooled May+June: около `58 721`.

Сравнение с предыдущими отправками даёт ориентир около **76k на публичной части лидерборда

## 1. Что нужно для запуска

Для запуска на нужны только:

- `train.csv`, `test.csv`, `sample_submission.csv` рядом с notebook или в папке `data/`;
- Python 3.12 и библиотеки из списка ниже.

Установка библиотек:

```bash
pip install lightgbm==4.6.0 numpy==2.5.0 pandas==3.0.3 scipy==1.18.0 ipython==9.15.0 ipykernel==7.3.0 jupyter-core==5.9.1 jupyter-client==8.9.1
```

## 2. Параметры моделей и зачем они нужны

Основная модель может построить до 4200 деревьев. При проверке по месяцам обучение
останавливается, если качество долго не улучшается. Для окончательной модели берётся
типичное число деревьев, полученное на этих проверках — 1965.

Модель-поправка специально сделана проще и жёстче ограничена. Это снижает риск того,
что она запомнит случайные особенности нескольких клиентов с очень высоким доходом.

Основные защитные меры:

- абсолютная ошибка вместо квадратичной меньше зависит от единичных очень больших значений;
- дереву запрещено создавать слишком маленькие группы клиентов;
- модель использует не все строки и столбцы в каждом дереве, что уменьшает переобучение;
- предлагаемая поправка ограничивается диапазоном от −80 тыс. до +180 тыс.;
- в итоговый прогноз добавляется только половина этой поправки.

Ниже показаны точные технические параметры, необходимые для повторного запуска.

In [12]:
import json
import sys
from pathlib import Path
from typing import Any

import lightgbm as lgb
import numpy as np
import pandas as pd
from IPython.display import display

ID_COL = "id"
TARGET = "target"
WEIGHT = "w"
TARGET_MIN = 20_000.0
TARGET_MAX = 1_500_000.0
NA_VALUES = ["", "nan", "None", "NONE", "null", "NULL"]

KEEP_AS_CATEGORY = {
    "gender",
    "adminarea",
    "city_smart_name",
    "dp_ewb_last_employment_position",
    "addrref",
    "dp_ewb_last_organization",
    "period_last_act_ad",
}
MAIN_EXCLUDED_FEATURES = {
    ID_COL,
    TARGET,
    WEIGHT,
    "dt",
    "first_salary_income",
}
FEATURE_FAMILIES = {
    "salary_income": [
        "salary", "incomeValue", "dp_payoutincomedata", "dp_ils", "first_salary_income"
    ],
    "turnover": ["turn_", "avg_cur_", "avg_debet_turn", "avg_credit_turn"],
    "balances": ["rur_amt", "balance", "curbal", "dda_rur", "total_rur"],
    "bki_credit": ["bki", "hdb_", "loan", "ovrd", "outstand", "limit"],
    "transactions": ["transaction_", "by_category", "amount_by_category", "avg_3m", "avg_6m"],
    "geo_demo": ["age", "gender", "adminarea", "city_smart_name", "label_"],
    "app_mobile": ["vert_", "mob_", "device_", "cnt", "sms", "called", "businessTel"],
}

START_DIR = Path.cwd().resolve()
required_files = ("train.csv", "test.csv", "sample_submission.csv")
data_candidates = [START_DIR, START_DIR / "data", START_DIR.parent]
DATA_DIR = next(
    (
        candidate
        for candidate in data_candidates
        if all((candidate / filename).exists() for filename in required_files)
    ),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Положите train.csv, test.csv и sample_submission.csv рядом с notebook "
        "или в подпапку data/."
    )


def read_data(data_dir: Path = DATA_DIR) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(
        data_dir / "train.csv",
        sep=";",
        decimal=",",
        na_values=NA_VALUES,
        low_memory=False,
    )
    test = pd.read_csv(
        data_dir / "test.csv",
        sep=";",
        decimal=",",
        na_values=NA_VALUES,
        low_memory=False,
    )
    sample = pd.read_csv(data_dir / "sample_submission.csv", sep=";", decimal=",")
    return train, test, sample


def weighted_absolute_error_sum(y_true: Any, y_pred: Any, weights: Any) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weights = np.asarray(weights, dtype=float)
    return float(np.sum(weights * np.abs(y_true - y_pred)))


def coerce_numeric_like(
    frame: pd.DataFrame,
    keep_as_category: set[str] = KEEP_AS_CATEGORY,
    threshold: float = 0.95,
) -> tuple[pd.DataFrame, list[str]]:
    result = frame.copy()
    converted = []
    for column in result.select_dtypes(include=["object", "string"]).columns:
        if column in keep_as_category or column == "dt":
            continue
        values = result[column].astype("string")
        numeric = pd.to_numeric(values.str.replace(",", ".", regex=False), errors="coerce")
        non_null = int(values.notna().sum())
        if non_null and numeric.notna().sum() / non_null >= threshold:
            result[column] = numeric
            converted.append(column)
    return result, converted


def parse_dt_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    dates = pd.to_datetime(result["dt"], errors="coerce")
    result["dt_month"] = dates.dt.month.astype("float")
    result["dt_quarter"] = dates.dt.quarter.astype("float")
    result["month_idx"] = (dates.dt.year - dates.dt.year.min()) * 12 + dates.dt.month
    return result


def columns_matching(columns: list[str], needles: list[str]) -> list[str]:
    lowered = {column: column.lower() for column in columns}
    return [
        column
        for column in columns
        if any(needle.lower() in lowered[column] for needle in needles)
    ]


def add_missing_indicators(
    train: pd.DataFrame,
    test: pd.DataFrame,
    candidate_features: list[str],
    min_missing: float = 0.50,
    max_indicators: int = 60,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    train = train.copy()
    test = test.copy()
    missing_rates = train[candidate_features].isna().mean().sort_values(ascending=False)
    selected = [
        column
        for column in missing_rates.index
        if missing_rates[column] >= min_missing
    ][:max_indicators]
    created = []
    for column in selected:
        name = f"miss__{column}"
        train[name] = train[column].isna().astype(np.int8)
        test[name] = test[column].isna().astype(np.int8)
        created.append(name)
    return train, test, created


def add_log_features(
    train: pd.DataFrame,
    test: pd.DataFrame,
    candidate_features: list[str],
    max_features: int = 80,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    train = train.copy()
    test = test.copy()
    numeric = train[candidate_features].select_dtypes(include=[np.number]).columns.tolist()
    money_columns = [
        column
        for column in numeric
        if any(
            token in column.lower()
            for token in (
                "amt", "amount", "sum", "salary", "income", "limit",
                "outstand", "payment", "turn", "balance"
            )
        )
    ]
    selected = (
        train[money_columns]
        .std(numeric_only=True)
        .sort_values(ascending=False)
        .index.tolist()[:max_features]
    )
    created = []
    for column in selected:
        name = f"log1p_abs__{column}"
        train[name] = np.sign(train[column]) * np.log1p(np.abs(train[column]))
        test[name] = np.sign(test[column]) * np.log1p(np.abs(test[column]))
        created.append(name)
    return train, test, created


def safe_ratio(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    return numerator / denominator.replace(0, np.nan)


def add_ratio_features(
    train: pd.DataFrame,
    test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    train = train.copy()
    test = test.copy()
    specifications = [
        ("ratio__avg_cur_db_to_cr_turn", "avg_cur_db_turn", "avg_cur_cr_turn"),
        ("ratio__turn_cur_db_to_cr_sum_v2", "turn_cur_db_sum_v2", "turn_cur_cr_sum_v2"),
        ("ratio__turn_cur_db_to_cr_avg_v2", "turn_cur_db_avg_v2", "turn_cur_cr_avg_v2"),
        ("ratio__total_balance_to_income", "total_rur_amt_cm_avg", "incomeValue"),
        ("ratio__curr_balance_to_income", "curr_rur_amt_cm_avg", "incomeValue"),
        ("ratio__loanacc_to_income", "loanacc_rur_amt_cm_avg", "incomeValue"),
        ("ratio__bki_limit_to_income", "bki_total_max_limit", "incomeValue"),
        ("ratio__hdb_bki_limit_to_income", "hdb_bki_total_max_limit", "incomeValue"),
        ("ratio__hdb_outstand_to_limit", "hdb_outstand_sum", "hdb_bki_total_max_limit"),
        ("ratio__hdb_relend_outstand_to_limit", "hdb_relend_outstand_sum", "hdb_bki_total_max_limit"),
        ("ratio__cash_to_all_6m", "avg_6m_money_transactions", "avg_6m_all"),
        ("ratio__restaurants_to_all_6m", "avg_6m_restaurants", "avg_6m_all"),
        ("ratio__travel_to_all_6m", "avg_6m_travel", "avg_6m_all"),
    ]
    created = []
    for name, numerator, denominator in specifications:
        if numerator in train.columns and denominator in train.columns:
            train[name] = safe_ratio(train[numerator], train[denominator])
            test[name] = safe_ratio(test[numerator], test[denominator])
            created.append(name)
    return train, test, created


def add_family_aggregates(
    train: pd.DataFrame,
    test: pd.DataFrame,
    candidate_features: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    train = train.copy()
    test = test.copy()
    numeric = train[candidate_features].select_dtypes(include=[np.number]).columns.tolist()
    created = []
    for family, needles in FEATURE_FAMILIES.items():
        columns = columns_matching(numeric, needles)
        columns = [column for column in columns if column in test.columns]
        if len(columns) < 2:
            continue
        prefix = f"family__{family}"
        train[f"{prefix}__non_missing_cnt"] = train[columns].notna().sum(axis=1)
        test[f"{prefix}__non_missing_cnt"] = test[columns].notna().sum(axis=1)
        train[f"{prefix}__mean"] = train[columns].mean(axis=1)
        test[f"{prefix}__mean"] = test[columns].mean(axis=1)
        train[f"{prefix}__max"] = train[columns].max(axis=1)
        test[f"{prefix}__max"] = test[columns].max(axis=1)
        created.extend(
            [f"{prefix}__non_missing_cnt", f"{prefix}__mean", f"{prefix}__max"]
        )
    return train, test, created


def make_time_folds(train: pd.DataFrame) -> list[tuple[pd.Index, pd.Index]]:
    months = sorted(train["dt"].dropna().unique())
    folds = []
    for valid_month in months[3:]:
        train_index = train.index[train["dt"] < valid_month]
        valid_index = train.index[train["dt"] == valid_month]
        if len(train_index) and len(valid_index):
            folds.append((train_index, valid_index))
    return folds


def prepare_datasets(
    train_raw: pd.DataFrame,
    test_raw: pd.DataFrame,
    feature_set: str = "engineered_v1",
    add_target_encoding: bool = False,
) -> dict[str, Any]:
    if feature_set != "engineered_v1" or add_target_encoding:
        raise ValueError("Этот автономный notebook поддерживает только engineered_v1 без target encoding")

    train, _ = coerce_numeric_like(train_raw)
    test, _ = coerce_numeric_like(test_raw)
    train = parse_dt_features(train)
    test = parse_dt_features(test)

    base_features = [
        column
        for column in test.columns
        if column in train.columns and column not in MAIN_EXCLUDED_FEATURES
    ]
    all_test_missing = [column for column in base_features if test[column].isna().all()]
    base_features = [column for column in base_features if column not in all_test_missing]

    created_features = {}
    train, test, created_features["missing_indicators"] = add_missing_indicators(
        train, test, base_features
    )
    train, test, created_features["log_features"] = add_log_features(
        train, test, base_features
    )
    train, test, created_features["ratios"] = add_ratio_features(train, test)
    train, test, created_features["family_aggregates"] = add_family_aggregates(
        train, test, base_features
    )

    engineered_features = [
        column
        for columns in created_features.values()
        for column in columns
        if column in train.columns and column in test.columns
    ]
    allowed = set(base_features) | set(engineered_features)
    features = [
        column
        for column in test.columns
        if column in allowed and column not in MAIN_EXCLUDED_FEATURES
    ]
    features = [column for column in features if not test[column].isna().all()]
    folds = make_time_folds(train)

    categorical_features = (
        train[features]
        .select_dtypes(include=["object", "string"])
        .columns.tolist()
    )
    for column in categorical_features:
        train[column] = train[column].astype("string").fillna("__NA__")
        test[column] = test[column].astype("string").fillna("__NA__")

    return {
        "train": train,
        "test": test,
        "features": features,
        "cat_features": categorical_features,
        "folds": folds,
        "created_features": created_features,
    }


def prepare_lightgbm_frames(
    train: pd.DataFrame,
    test: pd.DataFrame,
    features: list[str],
    categorical_features: list[str],
    categorical_mode: str = "codes",
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    lightgbm_train = train[features].copy()
    lightgbm_test = test[features].copy()
    native_categorical = []
    for column in categorical_features:
        if column not in lightgbm_train.columns:
            continue
        all_values = (
            pd.concat([lightgbm_train[column], lightgbm_test[column]], axis=0)
            .astype("string")
            .fillna("__NA__")
        )
        categories = pd.Index(all_values.unique())
        if categorical_mode == "category":
            lightgbm_train[column] = pd.Categorical(
                lightgbm_train[column].astype("string").fillna("__NA__"),
                categories=categories,
            )
            lightgbm_test[column] = pd.Categorical(
                lightgbm_test[column].astype("string").fillna("__NA__"),
                categories=categories,
            )
            native_categorical.append(column)
        else:
            mapping = {value: index for index, value in enumerate(categories)}
            lightgbm_train[column] = (
                lightgbm_train[column]
                .astype("string")
                .fillna("__NA__")
                .map(mapping)
                .astype("int32")
            )
            lightgbm_test[column] = (
                lightgbm_test[column]
                .astype("string")
                .fillna("__NA__")
                .map(mapping)
                .astype("int32")
            )
    return lightgbm_train, lightgbm_test, native_categorical


def make_submission(
    sample: pd.DataFrame,
    test: pd.DataFrame,
    prediction: np.ndarray,
    path: Path,
) -> pd.DataFrame:
    submission = sample[[ID_COL]].merge(
        pd.DataFrame({ID_COL: test[ID_COL].to_numpy(), "predict": prediction}),
        on=ID_COL,
        how="left",
    )
    assert list(submission.columns) == [ID_COL, "predict"]
    assert len(submission) == len(sample)
    assert submission["predict"].notna().all()
    assert set(submission[ID_COL]) == set(sample[ID_COL])
    path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(path, sep=";", decimal=",", index=False)
    return submission
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

OUTPUT_DIR = DATA_DIR / "outputs" / "simple_76k_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / "submission_simple_76k.csv"

BASE_PARAMS = {
    "objective": "regression_l1",
    "n_estimators": 4200,
    "learning_rate": 0.01841635833416662,
    "num_leaves": 123,
    "max_depth": -1,
    "min_child_samples": 74,
    "subsample": 0.8927531198095575,
    "subsample_freq": 1,
    "colsample_bytree": 0.8298737721743058,
    "reg_alpha": 0.3120408127066865,
    "reg_lambda": 6.826971955402957,
    "max_bin": 255,
    "random_state": 49,
    "n_jobs": 4,
    "verbosity": -1,
}
RESIDUAL_PARAMS_NB = {
    "objective": "regression_l1",
    "n_estimators": 640,                      # из study.best_params
    "learning_rate": 0.017749871775782373,    # из study.best_params
    "num_leaves": 38,                         # из study.best_params
    "min_child_samples": 223,                 # из study.best_params
    "subsample": 0.9429737206725748,          # из study.best_params
    "subsample_freq": 1,
    "colsample_bytree": 0.7215635393481526,   # из study.best_params
    "reg_alpha": 4.6365595702252635,          # из study.best_params
    "reg_lambda": 15.687473315419986,         # из study.best_params
    "random_state": 42,
    "n_jobs": 2,
    "verbosity": -1,
}

# Также зафиксируйте найденный лучший коэффициент поправки:
FINAL_STRENGTH = 0.94

COMPACT_FEATURES = [
    "salary_6to12m_avg", "incomeValue", "dp_ils_avg_salary_1y",
    "dp_ils_avg_salary_2y", "dp_ils_avg_salary_3y",
    "dp_payoutincomedata_payout_avg_3_month",
    "dp_payoutincomedata_payout_avg_6_month",
    "dp_payoutincomedata_payout_avg_prev_year",
    "hdb_bki_total_max_limit", "hdb_bki_total_cc_max_limit",
    "hdb_bki_total_pil_max_limit", "hdb_bki_active_cc_max_limit",
    "hdb_bki_active_cc_max_outstand", "hdb_outstand_sum", "bki_total_max_limit",
    "turn_cur_cr_sum_v2", "turn_cur_db_sum_v2", "avg_cur_cr_turn",
    "avg_cur_db_turn", "avg_6m_all", "avg_6m_money_transactions",
    "total_rur_amt_cm_avg", "curr_rur_amt_cm_avg", "loanacc_rur_amt_cm_avg",
    "incomeValueCategory", "age",
    "family__salary_income__mean", "family__salary_income__max",
    "family__salary_income__non_missing_cnt", "family__turnover__mean",
    "family__turnover__max", "family__balances__mean", "family__balances__max",
    "family__bki_credit__mean", "family__bki_credit__max",
    "family__transactions__mean", "family__transactions__max",
    "ratio__avg_cur_db_to_cr_turn", "ratio__turn_cur_db_to_cr_sum_v2",
    "ratio__hdb_bki_limit_to_income", "ratio__hdb_outstand_to_limit",
    "ratio__total_balance_to_income",
    "bki_total_il_max_limit", "hdb_relend_outstand_sum",
    "turn_cur_cr_avg_act_v2", "turn_cur_db_avg_act_v2",
    "avg_amount_daily_transactions_90d", "per_capita_income_rur_amt",
    "smsInWavg6m", "cntRegionTripsWavg1m", "gender",
    "avg_by_category__amount__sum__cashflowcategory_name__vydacha_nalichnyh_v_bankomate",
    "avg_by_category__amount__sum__cashflowcategory_name__elektronnye_dengi",
    "by_category__amount__sum__eoperation_type_name__perevod_po_nomeru_telefona",
    "by_category__amount__sum__eoperation_type_name__ishodjaschij_bystryj_platezh_sbp",
    "by_category__amount__sum__eoperation_type_name__vhodjaschij_bystryj_platezh_sbp",
    "curbal_usd_amt_cm_avg", "diff_avg_cr_db_turn", "hdb_other_outstand_sum",
    "turn_cur_cr_max_v2", "turn_cur_db_max_v2",
    "transaction_category_supermarket_percent_cnt_2m",
    "transaction_category_restaurants_percent_cnt_2m",
]
SALARY_PROXY_SOURCES = [
    "salary_6to12m_avg",
    "incomeValue",
    "dp_ils_avg_salary_1y",
    "dp_payoutincomedata_payout_avg_6_month",
]
SALARY_PROXY_FEATURES = [
    "proxy__salary_consensus",
    "proxy__salary_confidence",
    "proxy__salary_confident_gap_to_base",
]
CORRECTION_MIN, CORRECTION_MAX = -80_000.0, 180_000.0
# FINAL_STRENGTH = 0.6
EXPECTED_BASE_WMAE = np.array([63164.4523, 59606.0031, 59312.4749])
EXPECTED_BASE_ITERATIONS = [1285, 2713, 1965]
EXPECTED_RESIDUAL_WMAE = np.array([58680.1793, 58761.1520])

print(
    {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "lightgbm": lgb.__version__,
    }
)
print("Base parameters")
display(pd.Series(BASE_PARAMS, name="value").to_frame())
print("Residual parameters")
display(pd.Series(RESIDUAL_PARAMS_NB, name="value").to_frame())

{'python': '3.11.9', 'numpy': '2.2.3', 'pandas': '2.2.3', 'lightgbm': '4.6.0'}
Base parameters


,value
objective,regression_l1
n_estimators,4200
learning_rate,0.0184
num_leaves,123
max_depth,-1
min_child_samples,74
subsample,0.8928
subsample_freq,1
colsample_bytree,0.8299
reg_alpha,0.3120


Residual parameters


,value
objective,regression_l1
n_estimators,640
learning_rate,0.0177
num_leaves,38
min_child_samples,223
subsample,0.9430
subsample_freq,1
colsample_bytree,0.7216
reg_alpha,4.6366
reg_lambda,15.6875


## 3. Данные

Сначала читаем три исходных файла и проверяем, что количество строк ожидаемое, `id` не
повторяются, а ответы и веса в обучающих данных заполнены. Также проверяем, что итоговый
файл будет содержать клиентов в том же порядке, что и `sample_submission.csv`.

In [13]:
train_raw, test_raw, sample_submission = read_data()

assert len(train_raw) == 76_786
assert len(test_raw) == 73_214
assert len(sample_submission) == len(test_raw)
assert train_raw[ID_COL].is_unique and test_raw[ID_COL].is_unique
assert sample_submission[ID_COL].is_unique
assert set(sample_submission[ID_COL]) == set(test_raw[ID_COL])
assert train_raw[TARGET].notna().all()
assert (pd.to_numeric(train_raw[WEIGHT], errors="raise") > 0).all()
assert pd.to_datetime(train_raw["dt"], errors="raise").notna().all()
assert pd.to_datetime(test_raw["dt"], errors="raise").notna().all()

display(
    pd.DataFrame(
        {
            "rows": [len(train_raw), len(test_raw)],
            "columns": [train_raw.shape[1], test_raw.shape[1]],
            "months": [
                pd.to_datetime(train_raw["dt"]).nunique(),
                pd.to_datetime(test_raw["dt"]).nunique(),
            ],
        },
        index=["train", "test"],
    )
)

,rows,columns,months
train,76786,224,6
test,73214,222,5


## 4. Создание признаков для основной модели

**Признак** — это число или категория, которые модель использует для прогноза. Основная
модель получает 396 признаков. Новые признаки создаются по заранее заданным правилам и
не используют правильные ответы из столбца `target`.

| Семейство | Что строится | Зачем |
|---|---|---|
| 222 исходных и календарных | Доступные поля плюс месяц, квартал и номер месяца | Сохраняют исходные сведения о клиенте и изменение данных со временем |
| 60 флагов отсутствия | Для часто пустых полей создаётся признак «значение отсутствует» | Иногда отсутствие банковского источника само по себе полезно для прогноза |
| 80 log признаков | Очень большие по модулю числа переводятся в более компактный масштаб, знак сохраняется | Миллионные значения меньше подавляют различия между обычными клиентами |
| 13 отношений величин | Например, долг к лимиту или остаток к доходу | Показывают финансовую нагрузку, а не только размер суммы |
| 21 сводный признак | Для семи групп полей считаются количество заполненных значений, среднее и максимум | Объединяют несколько похожих источников в более устойчивую общую оценку |

`id`, дата, правильный ответ, вес строки и недоступный в test `first_salary_income` не
передаются модели. Текстовые категории заменяются стабильными целыми кодами: одинаковая
категория получает одинаковый код в train и test.

In [14]:
from sklearn.model_selection import KFold

# 1. Загружаем подготовленные датасеты
base_prepared = prepare_datasets(
    train_raw,
    test_raw,
    feature_set="engineered_v1",
    add_target_encoding=False,
)
base_train = base_prepared["train"]
base_test = base_prepared["test"]
base_features = list(base_prepared["features"])
base_cat_features = list(base_prepared["cat_features"])

forbidden = {ID_COL, TARGET, WEIGHT, "dt", "first_salary_income"}
assert not (set(base_features) & forbidden)

# 2. СНАЧАЛА готовим числовые матричные представления (переводим строки в коды int32), 
# чтобы ни один алгоритм LightGBM не упал из-за типов string
auto_cat_features = list(
    set(base_cat_features) | 
    set(base_train[base_features].select_dtypes(include=["object", "string", "category"]).columns)
)

x_base_train, x_base_test, _ = prepare_lightgbm_frames(
    base_train,
    base_test,
    base_features,
    auto_cat_features,
    categorical_mode="codes",
)

# 3. --- ИМПУТАЦИЯ first_salary_income (работаем с безопасными x_base_train / x_base_test) ---
print("Создаем модель для предсказания отсутствующего first_salary_income...")
train_has_salary = base_train["first_salary_income"].notna()
impute_features = [f for f in base_features if f != "first_salary_income"]

train_salary_preds = np.full(len(base_train), np.nan)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for tr_idx, val_idx in kf.split(base_train):
    mask_tr = train_has_salary.iloc[tr_idx]
    if mask_tr.sum() < 100:
        continue
    imp_model = lgb.LGBMRegressor(
        objective="regression_l1", n_estimators=300, learning_rate=0.03, random_state=42, verbosity=-1
    )
    # Используем числовые матрицы x_base_train
    imp_model.fit(
        x_base_train.loc[tr_idx[mask_tr], impute_features],
        base_train.loc[tr_idx[mask_tr], "first_salary_income"]
    )
    train_salary_preds[val_idx] = imp_model.predict(x_base_train.loc[val_idx, impute_features])

# Обучаем финальную модель импутации на всем трейне
full_imp_model = lgb.LGBMRegressor(
    objective="regression_l1", n_estimators=300, learning_rate=0.03, random_state=42, verbosity=-1
)
full_imp_model.fit(
    x_base_train.loc[train_has_salary, impute_features],
    base_train.loc[train_has_salary, "first_salary_income"]
)

# Заполняем пропуски в train и предсказываем для test
base_train["imputed__first_salary_income"] = base_train["first_salary_income"].fillna(
    pd.Series(train_salary_preds, index=base_train.index)
)
base_test["imputed__first_salary_income"] = full_imp_model.predict(x_base_test[impute_features])

base_train["miss__first_salary_income"] = base_train["first_salary_income"].isna().astype(np.int8)
base_test["miss__first_salary_income"] = 1

# Добавляем новые фичи в общий список признаков
new_impute_cols = ["imputed__first_salary_income", "miss__first_salary_income"]
base_features.extend(new_impute_cols)

# Добавляем новые колонки в уже готовые числовые фреймы x_base_train / x_base_test
x_base_train["imputed__first_salary_income"] = base_train["imputed__first_salary_income"]
x_base_train["miss__first_salary_income"] = base_train["miss__first_salary_income"]
x_base_test["imputed__first_salary_income"] = base_test["imputed__first_salary_income"]
x_base_test["miss__first_salary_income"] = base_test["miss__first_salary_income"]

assert list(x_base_train.columns) == list(x_base_test.columns) == base_features

feature_engineering_summary = pd.DataFrame(
    [
        {"family": name, "created": len(columns)}
        for name, columns in base_prepared["created_features"].items()
    ]
)
display(feature_engineering_summary)
print(f"Base matrices: train={x_base_train.shape}, test={x_base_test.shape}")

Создаем модель для предсказания отсутствующего first_salary_income...


,family,created
0,missing_indicators,60
1,log_features,80
2,ratios,13
3,family_aggregates,21


Base matrices: train=(76786, 398), test=(73214, 398)


## 5. Обучение и проверка основной модели

Test относится к следующему периоду, поэтому случайно перемешивать строки нельзя — такая
проверка была бы слишком лёгкой. Мы имитируем реальную работу модели: учим её на прошлых
месяцах и проверяем на следующем:

- January–March → April;
- January–April → May;
- January–May → June.

Такой прогноз называется **OOF-прогнозом**: модель, которая прогнозирует конкретную
строку, не обучалась на этой строке и не видела её правильный ответ. Для April–June
получаем 47 265 таких прогнозов. Затем модель-поправка учится именно на этих честно
полученных ошибках.

После проверки основная модель обучается на всём train. Число деревьев выбирается как
середина трёх значений, полученных при проверке.

Ограничение оценки: параметры модели раньше подбирались с учётом June. Поэтому эти
результаты хорошо воспроизводят процесс выбора модели, но не являются совершенно новой
независимой проверкой.

In [15]:
base_oof = np.full(len(base_train), np.nan, dtype=float)
base_score_rows = []

for fold_id, (train_idx, valid_idx) in enumerate(base_prepared["folds"], start=1):
    valid_month = pd.Timestamp(base_train.loc[valid_idx, "dt"].iloc[0])
    model = lgb.LGBMRegressor(**BASE_PARAMS)
    model.fit(
        x_base_train.loc[train_idx, base_features],
        base_train.loc[train_idx, TARGET],
        sample_weight=base_train.loc[train_idx, WEIGHT],
        eval_set=[(x_base_train.loc[valid_idx, base_features], base_train.loc[valid_idx, TARGET])],
        eval_sample_weight=[base_train.loc[valid_idx, WEIGHT]],
        eval_metric="l1",
        callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)],
    )
    prediction = np.clip(model.predict(x_base_train.loc[valid_idx, base_features]), TARGET_MIN, TARGET_MAX)
    base_oof[valid_idx] = prediction
    error_sum = weighted_absolute_error_sum(
        base_train.loc[valid_idx, TARGET], prediction, base_train.loc[valid_idx, WEIGHT]
    )
    weight_sum = float(base_train.loc[valid_idx, WEIGHT].sum())
    base_score_rows.append(
        {
            "fold": fold_id,
            "valid_month": valid_month.strftime("%Y-%m-%d"),
            "n_train": len(train_idx),
            "n_valid": len(valid_idx),
            "wmae": error_sum / weight_sum,
            "weighted_abs_error_sum": error_sum,
            "weight_sum": weight_sum,
            "best_iteration": int(model.best_iteration_),
        }
    )
del model
base_scores = pd.DataFrame(base_score_rows)
observed_wmae = base_scores["wmae"].to_numpy(dtype=float)
observed_iterations = base_scores["best_iteration"].astype(int).tolist()
if not np.allclose(observed_wmae, EXPECTED_BASE_WMAE, rtol=0, atol=100.0):
    print("Предупреждение: качество base заметно отличается от контрольного запуска")
if observed_iterations != EXPECTED_BASE_ITERATIONS:
    print(
        "Предупреждение: early stopping выбрал другие числа деревьев:",
        observed_iterations,
    )
assert np.isfinite(base_oof).sum() == 47_265

# Число деревьев финальной модели зафиксировано, чтобы результат не зависел
# от небольших различий early stopping между компьютерами.
base_final_iterations = 1965
final_base_params = {**BASE_PARAMS, "n_estimators": base_final_iterations}
base_model = lgb.LGBMRegressor(**final_base_params)
base_model.fit(
    x_base_train[base_features],
    base_train[TARGET],
    sample_weight=base_train[WEIGHT],
)
base_test_prediction = np.clip(
    base_model.predict(x_base_test[base_features]),
    TARGET_MIN,
    TARGET_MAX,
)

base_oof_frame = base_train[[ID_COL, "dt", TARGET, WEIGHT]].copy()
base_oof_frame["prediction"] = base_oof
base_test_frame = pd.DataFrame({ID_COL: base_test[ID_COL], "prediction": base_test_prediction})
display(base_scores)

Предупреждение: качество base заметно отличается от контрольного запуска
Предупреждение: early stopping выбрал другие числа деревьев: [1287, 1714, 1712]


,fold,valid_month,n_train,n_valid,wmae,weighted_abs_error_sum,weight_sum,best_iteration
0,1,2024-04-30,29521,14858,"61,925.8975","517,225,086.8091","8,352.3228",1287
1,2,2024-05-31,44379,16193,"59,207.1645","537,370,905.4472","9,076.1128",1714
2,3,2024-06-30,60572,16214,"59,884.3510","541,509,885.2946","9,042.5942",1712


In [16]:
import lightgbm as lgb
import numpy as np
import pandas as pd

# Генерируем 8 порогов равномерно в логарифмическом масштабе от 50k до 1M
raw_thresholds = np.geomspace(50_000, 1_000_000, num=8)

# Округляем до десятков тысяч для стабильности и красоты признаков
THRESHOLDS = sorted(list(set([int(round(t, -4)) for t in raw_thresholds])))

print(f"Пороги на основе геом. прогрессии: {THRESHOLDS}")

clf_oof = {t: np.full(len(base_train), np.nan, dtype=float) for t in THRESHOLDS}
clf_test_preds = {t: np.zeros(len(base_test), dtype=float) for t in THRESHOLDS}

# Параметры для быстрых классификаторов
CLF_PARAMS = {
    "objective": "binary",
    "n_estimators": 800,
    "learning_rate": 0.03,
    "num_leaves": 31,
    "random_state": 42,
    "n_jobs": 4,
    "verbosity": -1,
}

print(f"Обучаем КФР ({len(THRESHOLDS)} бинарных порогов)...")

for t in THRESHOLDS:
    print(f"Обучение классификатора: вероятность дохода > {t // 1000}k...")
    y_train_bin = (base_train[TARGET] > t).astype(int)
    
    for fold_id, (train_idx, valid_idx) in enumerate(base_prepared["folds"]):
        model = lgb.LGBMClassifier(**CLF_PARAMS)
        model.fit(
            x_base_train.loc[train_idx, base_features],
            y_train_bin.iloc[train_idx],
            eval_set=[(x_base_train.loc[valid_idx, base_features], y_train_bin.iloc[valid_idx])],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(100, verbose=False)]
        )
        clf_oof[t][valid_idx] = model.predict_proba(x_base_train.loc[valid_idx, base_features])[:, 1]
        
    final_model = lgb.LGBMClassifier(**CLF_PARAMS)
    final_model.fit(x_base_train[base_features], y_train_bin)
    clf_test_preds[t] = final_model.predict_proba(x_base_test[base_features])[:, 1]
    
print("Готово! КФР вероятности (log-scale) рассчитаны.")

Пороги на основе геом. прогрессии: [50000, 80000, 120000, 180000, 280000, 420000, 650000, 1000000]
Обучаем КФР (8 бинарных порогов)...
Обучение классификатора: вероятность дохода > 50k...
Обучение классификатора: вероятность дохода > 80k...
Обучение классификатора: вероятность дохода > 120k...
Обучение классификатора: вероятность дохода > 180k...
Обучение классификатора: вероятность дохода > 280k...
Обучение классификатора: вероятность дохода > 420k...
Обучение классификатора: вероятность дохода > 650k...
Обучение классификатора: вероятность дохода > 1000k...
Готово! КФР вероятности (log-scale) рассчитаны.


## 6. Признаки для модели-поправки

**Модель-поправка** предсказывает не сам доход, а ошибку основной модели:
`фактический доход − основной прогноз`. Положительное значение означает, что основная
модель недооценила доход, отрицательное — переоценила. Для этой задачи достаточно 73
признаков. Они перечислены по понятным группам ниже.

| Семейство | Признаки | Что они объясняют |
|---|---|---|
| Оценки дохода (8) | `salary_6to12m_avg`, `incomeValue`, `dp_ils_avg_salary_1y/2y/3y`, `dp_payoutincomedata_payout_avg_3_month/6_month/prev_year` | Несколько независимых оценок регулярного дохода |
| Кредитные данные (7) | Лимиты, остаток долга и максимальный доступный кредит | Помогают оценить подтверждённую банками платёжеспособность и долговую нагрузку |
| Движение денег и остатки (9) | Поступления, списания, средние обороты и остатки на счетах | Показывают реальный денежный масштаб клиента, даже если зарплата известна не полностью |
| Возраст и категория дохода (2) | `incomeValueCategory`, `age` | Дают общий жизненный и доходный контекст |
| Сводные признаки (11) | Среднее, максимум и число заполненных полей по группам дохода, оборотов, остатков, кредитов и транзакций | Объединяют несколько похожих измерений одного клиента |
| Отношения величин (5) | Например, списания к поступлениям, долг к лимиту, остаток к доходу | Показывают нагрузку и структуру денег, а не только абсолютные суммы |
| Дополнительные сведения (9) | Ещё несколько кредитных, ежедневных, региональных и поведенческих показателей | Закрывают информацию, которой нет в основных группах |
| Транзакции и признаки высокого дохода (12) | Снятие наличных, электронные деньги, переводы, СБП, валютный остаток, максимальные обороты, доли супермаркетов и ресторанов | Помогают различать образ расходов и редкие обеспеченные профили |
| Прогноз основной модели (7) | Сам прогноз, его сжатая версия и флаги уровней 100/150/250/400/600 тыс. | Позволяют делать разную поправку для обычного и высокого диапазона доходов |
| Объединённая зарплата (3) | `salary_consensus`, `salary_confidence`, `salary_confident_gap_to_base` | Сравнивают несколько зарплатных источников с основной моделью без обучения третьей модели |

У одного клиента может быть до четырёх оценок зарплаты:
`salary_6to12m_avg`, `incomeValue`, `dp_ils_avg_salary_1y`,
`dp_payoutincomedata_payout_avg_6_month`. Отдельный источник может отсутствовать или
ошибаться, поэтому мы не доверяем одному числу. Берём центральное значение — медиану.
Она меньше зависит от единичной аномалии. Используются только конечные положительные
значения; всё выше 2,5 млн заранее ограничивается. Если источников нет, эти признаки
остаются пустыми.

Три добавленных признака имеют разные роли:

- `proxy__salary_consensus` — объединённая оценка зарплаты;
- `proxy__salary_confidence` — степень доверия: она выше, когда источников достаточно и
  они дают близкие значения;
- `proxy__salary_confident_gap_to_base` — сравнение объединённой зарплаты с прогнозом
  основной модели. При слабом доверии это различие автоматически уменьшается.

Формулы:

\[
consensus=median(positive\ salary\ sources)
\]

\[
confidence=\min(n/3,1)\exp\{-median|\log(1+x_i)-\log(1+consensus)|\}
\]

\[
confident\_gap=confidence\cdot clip(\log(1+consensus)-\log(1+base),-2.5,2.5)
\]

Разница считается в логарифмическом масштабе: так модель сравнивает скорее относительное
расхождение, например «примерно в два раза», а не только разницу в денежных единицах.
В train используется OOF-прогноз основной модели, поэтому правильный ответ строки сюда
не просачивается.

In [17]:
# Базовые действия Блока 6
residual_train = base_prepared["train"].copy()
residual_test = base_prepared["test"].copy()

residual_train["base_pred"] = residual_train[ID_COL].map(
    base_oof_frame.set_index(ID_COL)["prediction"]
)
residual_test["base_pred"] = residual_test[ID_COL].map(
    base_test_frame.set_index(ID_COL)["prediction"]
)

# Мета-признаки базового прогноза (теперь используем наши динамические THRESHOLDS)
meta_features = ["meta__base_pred", "meta__log1p_base_pred"]
meta_features += [
    f"meta__base_pred_ge_{t // 1000}k" for t in THRESHOLDS
]

for frame in (residual_train, residual_test):
    base = frame["base_pred"].to_numpy(dtype=float)
    frame["meta__base_pred"] = base
    frame["meta__log1p_base_pred"] = np.log1p(np.clip(base, 0.0, None))
    for t in THRESHOLDS:
        frame[f"meta__base_pred_ge_{t // 1000}k"] = (base >= t).astype(np.int8)

# Зарплатные прокси-признаки
for frame in (residual_train, residual_test):
    salary_values = frame[SALARY_PROXY_SOURCES].apply(pd.to_numeric, errors="coerce")
    salary_values = salary_values.where(
        salary_values.gt(0.0) & np.isfinite(salary_values)
    ).clip(upper=2_500_000.0)
    consensus = salary_values.median(axis=1, skipna=True)
    source_count = salary_values.notna().sum(axis=1).astype(float)
    log_values = np.log1p(salary_values)
    log_consensus = np.log1p(consensus)
    disagreement = log_values.sub(log_consensus, axis=0).abs().median(axis=1, skipna=True)
    confidence = np.minimum(source_count / 3.0, 1.0) * np.exp(-disagreement)
    log_gap = (
        log_consensus - np.log1p(frame["base_pred"].clip(lower=0.0))
    ).clip(-2.5, 2.5)
    frame["proxy__salary_consensus"] = consensus
    frame["proxy__salary_confidence"] = confidence
    frame["proxy__salary_confident_gap_to_base"] = confidence * log_gap

# Внедряем динамические вероятности КФР
cdf_features = []
for t in THRESHOLDS:
    col_name = f"prob_gt_{t // 1000}k"
    residual_train[col_name] = clf_oof[t]
    residual_test[col_name] = clf_test_preds[t]
    cdf_features.append(col_name)

# Дополнительные зарплатные признаки импутации
extra_salary_feat = ["imputed__first_salary_income", "miss__first_salary_income"]

# Убедимся, что они есть в датафреймах residual_train / residual_test
for frame in (residual_train, residual_test):
    frame["imputed__first_salary_income"] = base_train["imputed__first_salary_income"] if frame is residual_train else base_test["imputed__first_salary_income"]
    frame["miss__first_salary_income"] = base_train["miss__first_salary_income"] if frame is residual_train else base_test["miss__first_salary_income"]

# Собираем все признаки воедино
final_residual_features = COMPACT_FEATURES + meta_features + SALARY_PROXY_FEATURES + cdf_features + extra_salary_feat

# Обновляем словарь семейств
residual_families = {
    "direct_income": COMPACT_FEATURES[0:8],
    "credit_capacity": COMPACT_FEATURES[8:15],
    "cashflow_balances": COMPACT_FEATURES[15:24],
    "demography": COMPACT_FEATURES[24:26],
    "row_aggregates": COMPACT_FEATURES[26:37],
    "financial_ratios": COMPACT_FEATURES[37:42],
    "stable_signals": COMPACT_FEATURES[42:51],
    "transactions_tail": COMPACT_FEATURES[51:63],
    "base_meta": meta_features,
    "salary_consensus": SALARY_PROXY_FEATURES,
    "cdf_probs": cdf_features,
    "salary_imputed": extra_salary_feat,
}

feature_to_family = {
    feature: family
    for family, features in residual_families.items()
    for feature in features
}
assert set(feature_to_family) == set(final_residual_features)

residual_cat_features = [
    feature for feature in base_cat_features if feature in final_residual_features
]
x_residual_train, x_residual_test, _ = prepare_lightgbm_frames(
    residual_train,
    residual_test,
    final_residual_features,
    residual_cat_features,
    categorical_mode="codes",
)

family_summary = pd.DataFrame(
    [
        {"family": family, "n_features": len(features), "features": ", ".join(features)}
        for family, features in residual_families.items()
    ]
)
display(family_summary[["family", "n_features"]])

,family,n_features
0,direct_income,8
1,credit_capacity,7
2,cashflow_balances,9
3,demography,2
4,row_aggregates,11
5,financial_ratios,5
6,stable_signals,9
7,transactions_tail,12
8,base_meta,10
9,salary_consensus,3


## 7. Как признаки связаны с правильным ответом

**Корреляция Спирмена** показывает, сопровождаются ли большие значения признака обычно
большими или меньшими значениями дохода. Значение около `+1` означает одинаковый порядок,
около `−1` — противоположный, около `0` — устойчивой связи по порядку почти нет.

Корреляция не доказывает причинную связь и не используется здесь для дополнительного
отбора признаков. Мы только объясняем уже зафиксированную модель. Показываем:

- связь 396 признаков основной модели с фактическим доходом на постоянной выборке из
  30 000 строк;
- связь 73 признаков модели-поправки с фактическим доходом на April–June;
- связь этих 73 признаков с честно рассчитанной ошибкой основной модели за April–June.

Этот способ меньше зависит от единичных очень больших доходов. Для категорий результат
следует читать осторожно: присвоенный категории номер сам по себе ничего не означает.

In [18]:
correlation_index = base_train.sample(n=30_000, random_state=42).index
base_correlation_matrix = x_base_train.loc[correlation_index, base_features].replace(
    [np.inf, -np.inf], np.nan
)
base_target_correlation = base_correlation_matrix.corrwith(
    base_train.loc[correlation_index, TARGET],
    method="spearman",
)
created_family = {
    feature: family
    for family, features in base_prepared["created_features"].items()
    for feature in features
}
base_correlations = pd.DataFrame(
    {
        "feature": base_features,
        "family": [
            "date" if feature in {"dt_month", "dt_quarter", "month_idx"}
            else created_family.get(feature, "raw")
            for feature in base_features
        ],
        "spearman_target": [base_target_correlation.get(feature, np.nan) for feature in base_features],
    }
)
base_correlations["abs_target"] = base_correlations["spearman_target"].abs()

correlation_matrix = x_residual_train[final_residual_features].replace([np.inf, -np.inf], np.nan)

oof_mask = residual_train["base_pred"].notna()
target_correlation = correlation_matrix.loc[oof_mask].corrwith(
    residual_train.loc[oof_mask, TARGET],
    method="spearman",
)
oof_residual_target = (
    residual_train.loc[oof_mask, TARGET]
    - residual_train.loc[oof_mask, "base_pred"]
)
residual_correlation = correlation_matrix.loc[oof_mask].corrwith(
    oof_residual_target,
    method="spearman",
)

feature_correlations = pd.DataFrame(
    {
        "feature": final_residual_features,
        "family": [feature_to_family[feature] for feature in final_residual_features],
        "spearman_target": [target_correlation.get(feature, np.nan) for feature in final_residual_features],
        "spearman_oof_residual": [
            residual_correlation.get(feature, np.nan) for feature in final_residual_features
        ],
    }
)
feature_correlations["abs_target"] = feature_correlations["spearman_target"].abs()
feature_correlations["abs_oof_residual"] = feature_correlations["spearman_oof_residual"].abs()

print("Base: самые сильные связи с исходным target (фиксированная выборка 30k)")
display(
    base_correlations.sort_values("abs_target", ascending=False)
    .head(20)[["feature", "family", "spearman_target"]]
)
print("Residual features: самые сильные связи с исходным target")
display(
    feature_correlations.sort_values("abs_target", ascending=False)
    .head(20)[["feature", "family", "spearman_target"]]
)
print("Самые сильные связи с честной ошибкой base")
display(
    feature_correlations.sort_values("abs_oof_residual", ascending=False)
    .head(20)[["feature", "family", "spearman_oof_residual"]]
)

# del base_correlation_matrix, correlation_matrix
# del x_base_train, x_base_test, base_train, base_test, base_prepared

C:\Users\Shakhray.NS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pandas\core\nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]
C:\Users\Shakhray.NS\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pandas\core\nanops.py:1632: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


Base: самые сильные связи с исходным target (фиксированная выборка 30k)


,feature,family,spearman_target
1,salary_6to12m_avg,raw,0.9376
344,log1p_abs__salary_6to12m_avg,log_features,0.9376
349,log1p_abs__dp_payoutincomedata_payout_avg_6_month,log_features,0.7560
135,dp_payoutincomedata_payout_avg_6_month,raw,0.7560
348,log1p_abs__dp_payoutincomedata_payout_avg_3_month,log_features,0.7379
60,dp_payoutincomedata_payout_avg_3_month,raw,0.7379
327,log1p_abs__dp_payoutincomedata_payout_sum_3_month,log_features,0.7354
71,dp_payoutincomedata_payout_sum_3_month,raw,0.7354
75,dp_payoutincomedata_payout_max_6_month,raw,0.7286
334,log1p_abs__dp_payoutincomedata_payout_max_6_month,log_features,0.7286


Residual features: самые сильные связи с исходным target


,feature,family,spearman_target
0,salary_6to12m_avg,direct_income,0.9413
6,dp_payoutincomedata_payout_avg_6_month,direct_income,0.7726
5,dp_payoutincomedata_payout_avg_3_month,direct_income,0.7487
76,prob_gt_50k,cdf_probs,0.6819
64,meta__log1p_base_pred,base_meta,0.6734
63,meta__base_pred,base_meta,0.6734
2,dp_ils_avg_salary_1y,direct_income,0.6678
77,prob_gt_80k,cdf_probs,0.6664
7,dp_payoutincomedata_payout_avg_prev_year,direct_income,0.6496
3,dp_ils_avg_salary_2y,direct_income,0.6388


Самые сильные связи с честной ошибкой base


,feature,family,spearman_oof_residual
64,meta__log1p_base_pred,base_meta,-0.3361
63,meta__base_pred,base_meta,-0.3361
66,meta__base_pred_ge_80k,base_meta,-0.3321
65,meta__base_pred_ge_50k,base_meta,-0.2730
67,meta__base_pred_ge_120k,base_meta,-0.2710
75,proxy__salary_confident_gap_to_base,salary_consensus,0.2666
79,prob_gt_180k,cdf_probs,-0.2390
80,prob_gt_280k,cdf_probs,-0.2350
78,prob_gt_120k,cdf_probs,-0.2335
81,prob_gt_420k,cdf_probs,-0.2319


## 8. Обучение модели-поправки

Модель учится предсказывать ошибку: `фактический доход − OOF-прогноз основной модели`.
Проверка снова идёт по времени:

- обучаемся на ошибках April и проверяем прогноз ошибок May;
- обучаемся на ошибках April–May и проверяем прогноз ошибок June.

В notebook уже зафиксированы 73 признака и коэффициент поправки `0.50`; новый перебор
вариантов не выполняется. Слишком большая предложенная поправка жёстко ограничивается
диапазоном от −80 тыс. до +180 тыс.

Ограничение оценки: набор признаков и коэффициент ранее выбирались по May и June.
Поэтому это воспроизведение процесса выбора, а не новая независимая проверка.

In [19]:
# 1. Определяем даты и месяцы для OOF-валидации
dates = pd.to_datetime(residual_train["dt"])
oof_months = sorted(dates[residual_train["base_pred"].notna()].unique())

# 2. Рассчитываем OOF-метрики и собираем предсказания для подбора коэффициента
residual_score_rows = []
oof_y, oof_w, oof_base, oof_corr = [], [], [], []

for fold_id in range(1, len(oof_months)):
    train_months = oof_months[:fold_id]
    valid_month = oof_months[fold_id]
    train_mask = dates.isin(train_months) & residual_train["base_pred"].notna()
    valid_mask = dates.eq(valid_month) & residual_train["base_pred"].notna()
    
    residual_target = (
        residual_train.loc[train_mask, TARGET]
        - residual_train.loc[train_mask, "base_pred"]
    )

    # Обучаем модель с параметрами из RESIDUAL_PARAMS_NB
    fold_model = lgb.LGBMRegressor(**RESIDUAL_PARAMS_NB)
    fold_model.fit(
        x_residual_train.loc[train_mask, final_residual_features],
        residual_target,
        sample_weight=residual_train.loc[train_mask, WEIGHT],
    )
    
    raw_correction = np.clip(
        fold_model.predict(x_residual_train.loc[valid_mask, final_residual_features]),
        CORRECTION_MIN,
        CORRECTION_MAX,
    )
    base_prediction = residual_train.loc[valid_mask, "base_pred"].to_numpy(dtype=float)
    y_valid = residual_train.loc[valid_mask, TARGET].to_numpy(dtype=float)
    w_valid = residual_train.loc[valid_mask, WEIGHT].to_numpy(dtype=float)
    
    oof_y.extend(y_valid)
    oof_w.extend(w_valid)
    oof_base.extend(base_prediction)
    oof_corr.extend(raw_correction)
    
    base_error_sum = weighted_absolute_error_sum(y_valid, base_prediction, w_valid)
    weight_sum = float(w_valid.sum())
    
    residual_score_rows.append({
        "fold": fold_id,
        "valid_month": pd.Timestamp(valid_month).strftime("%Y-%m-%d"),
        "n_train": int(train_mask.sum()),
        "n_valid": int(valid_mask.sum()),
        "base_wmae": base_error_sum / weight_sum,
        "weight_sum": weight_sum,
        "y_valid": y_valid,
        "w_valid": w_valid,
        "base_prediction": base_prediction,
        "raw_correction": raw_correction
    })

# Переводим в массивы для поиска оптимальной силы поправки
arr_y = np.array(oof_y)
arr_w = np.array(oof_w)
arr_base = np.array(oof_base)
arr_corr = np.array(oof_corr)

def get_wmae(strength):
    pred = np.clip(arr_base + strength * arr_corr, TARGET_MIN, TARGET_MAX)
    return weighted_absolute_error_sum(arr_y, pred, arr_w) / arr_w.sum()

# 3. Подбираем лучший FINAL_STRENGTH перебором
best_strength = 0.50
best_wmae = float('inf')
for s in np.linspace(0.1, 1.0, 91):
    wmae = get_wmae(s)
    if wmae < best_wmae:
        best_wmae = wmae
        best_strength = s
        
FINAL_STRENGTH = round(float(best_strength), 2)
print(f"Финальный лучший коэффициент силы поправки: {FINAL_STRENGTH} (Pooled WMAE: {best_wmae:,.4f})")

# 4. Пересчитываем итоговые метрики по фолдам с найденным FINAL_STRENGTH
final_score_rows = []
for row in residual_score_rows:
    pred = np.clip(row["base_prediction"] + FINAL_STRENGTH * row["raw_correction"], TARGET_MIN, TARGET_MAX)
    error_sum = weighted_absolute_error_sum(row["y_valid"], pred, row["w_valid"])
    base_error_sum = row["base_wmae"] * row["weight_sum"]
    
    final_score_rows.append({
        "fold": row["fold"],
        "valid_month": row["valid_month"],
        "n_train": row["n_train"],
        "n_valid": row["n_valid"],
        "base_wmae": row["base_wmae"],
        "wmae": error_sum / row["weight_sum"],
        "gain": (base_error_sum - error_sum) / row["weight_sum"],
        "weighted_abs_error_sum": error_sum,
        "base_weighted_abs_error_sum": base_error_sum,
        "weight_sum": row["weight_sum"],
    })

residual_scores = pd.DataFrame(final_score_rows)

pooled_wmae = float(residual_scores["weighted_abs_error_sum"].sum() / residual_scores["weight_sum"].sum())
base_pooled_wmae = float(residual_scores["base_weighted_abs_error_sum"].sum() / residual_scores["weight_sum"].sum())

residual_choice = {
    "variant": "compact_proxy_optuna",
    "strength": FINAL_STRENGTH,
    "n_features": len(final_residual_features),
    "pooled_wmae": pooled_wmae,
    "base_pooled_wmae": base_pooled_wmae,
    "worst_fold_gain": float(residual_scores["gain"].min()),
}

display(residual_scores)
display(pd.Series(residual_choice, name="frozen_configuration").to_frame())

Финальный лучший коэффициент силы поправки: 0.98 (Pooled WMAE: 57,811.4181)


,fold,valid_month,n_train,n_valid,base_wmae,wmae,gain,weighted_abs_error_sum,base_weighted_abs_error_sum,weight_sum
0,1,2024-05-31,14858,16193,"59,207.1645","57,168.2551","2,038.9093","518,865,534.1346","537,370,905.4472","9,076.1128"
1,2,2024-06-30,31051,16214,"59,884.3510","58,456.9652","1,427.3858","528,602,615.1168","541,509,885.2946","9,042.5942"


,frozen_configuration
variant,compact_proxy_optuna
strength,0.9800
n_features,86
pooled_wmae,"57,811.4181"
base_pooled_wmae,"59,545.1313"
worst_fold_gain,"1,427.3858"


### Почему используется половина поправки

При выборе коэффициента требовалось, чтобы модель улучшала результат и в May, и в June.
Коэффициент `0.75` был лучше всего лишь примерно на 54 пункта средней ошибки, но немного
ухудшал June и сильнее зависел от нескольких клиентов с очень большим доходом.

Поэтому выбран коэффициент `0.50`: он использует только половину предложенной поправки
и даёт более устойчивый основной вариант.

## 9. Окончательное обучение и прогноз

Модель-поправка обучается на всех доступных честных ошибках за April–June. Итоговый
прогноз строится в четыре шага:

1. Основная модель оценивает доход.
2. Вторая модель предлагает поправку.
3. Поправка ограничивается диапазоном от −80 тыс. до +180 тыс., затем используется
   только её половина.
4. Готовый прогноз ограничивается диапазоном от 20 тыс. до 1,5 млн.

Эти ограничения не позволяют нескольким редким наблюдениям создать чрезмерно большой
прогноз. Та же операция кратко записана формулой:

\[
prediction=clip(base+0.50\cdot clip(residual,-80k,+180k),20k,1.5m)
\]

In [20]:
chosen_features = final_residual_features
chosen_strength = FINAL_STRENGTH
final_mask = residual_train["base_pred"].notna()

final_residual_params = {**RESIDUAL_PARAMS_NB, "n_estimators": 900}
residual_model = lgb.LGBMRegressor(**final_residual_params)
residual_model.fit(
    x_residual_train.loc[final_mask, chosen_features],
    residual_train.loc[final_mask, TARGET]
    - residual_train.loc[final_mask, "base_pred"],
    sample_weight=residual_train.loc[final_mask, WEIGHT],
)
raw_test_correction = np.clip(
    residual_model.predict(x_residual_test[chosen_features]),
    CORRECTION_MIN,
    CORRECTION_MAX,
)
final_prediction = np.clip(
    base_test_prediction + chosen_strength * raw_test_correction,
    TARGET_MIN,
    TARGET_MAX,
)
submission = make_submission(
    sample_submission,
    residual_test,
    final_prediction,
    SUBMISSION_PATH,
)

base_model_path = OUTPUT_DIR / "model_base.txt"
residual_model_path = OUTPUT_DIR / "model_residual.txt"
base_model.booster_.save_model(str(base_model_path))
residual_model.booster_.save_model(str(residual_model_path))

base_oof_frame.to_csv(OUTPUT_DIR / "base_oof.csv", index=False)
base_test_frame.to_csv(OUTPUT_DIR / "base_test_predictions.csv", index=False)
base_scores.to_csv(OUTPUT_DIR / "base_temporal_scores.csv", index=False)
residual_scores.to_csv(OUTPUT_DIR / "residual_temporal_scores.csv", index=False)
base_correlations.to_csv(OUTPUT_DIR / "base_feature_target_correlations.csv", index=False)
feature_correlations.to_csv(OUTPUT_DIR / "residual_feature_target_correlations.csv", index=False)
print(f"Submission: {SUBMISSION_PATH}")

Submission: C:\Users\Shakhray.NS\Desktop\Pract\outputs\simple_76k_notebook\submission_simple_76k.csv


## 10. Какие признаки модель использовала чаще всего

**Важность признака** показывает, насколько использование этого признака помогало
деревьям уменьшать ошибку во время обучения. Она помогает понять:

- на каких данных держится прогноз основной модели;
- использует ли модель-поправка три новых зарплатных признака.

Важность не доказывает, что признак является причиной высокого дохода. Кроме того,
похожие признаки могут делить важность между собой. Для модели-поправки мы показываем
как отдельные признаки, так и сумму по понятным группам.

In [21]:
base_importance = pd.DataFrame(
    {
        "feature": base_features,
        "gain": base_model.booster_.feature_importance(importance_type="gain"),
        "split": base_model.booster_.feature_importance(importance_type="split"),
    }
).sort_values(["gain", "split"], ascending=False)

residual_importance = pd.DataFrame(
    {
        "feature": chosen_features,
        "family": [feature_to_family[feature] for feature in chosen_features],
        "gain": residual_model.booster_.feature_importance(importance_type="gain"),
        "split": residual_model.booster_.feature_importance(importance_type="split"),
    }
).sort_values(["gain", "split"], ascending=False)
residual_family_importance = (
    residual_importance.groupby("family", as_index=False)
    .agg(gain=("gain", "sum"), split=("split", "sum"), n_features=("feature", "size"))
    .sort_values("gain", ascending=False)
)
residual_family_importance["gain_share"] = (
    residual_family_importance["gain"] / residual_family_importance["gain"].sum()
)

base_importance.to_csv(OUTPUT_DIR / "base_feature_importance.csv", index=False)
residual_importance.to_csv(OUTPUT_DIR / "residual_feature_importance.csv", index=False)
residual_family_importance.to_csv(OUTPUT_DIR / "residual_family_importance.csv", index=False)

print("Top-20 base features")
display(base_importance.head(20))
print("Top-20 residual features")
display(residual_importance.head(20))
print("Residual importance by family")
display(residual_family_importance)

Top-20 base features


,feature,gain,split
379,family__turnover__mean,"448,543.2536",1661
0,turn_cur_cr_avg_act_v2,"203,255.2725",1694
1,salary_6to12m_avg,"182,821.2881",3504
376,family__salary_income__mean,"137,004.9407",2275
385,family__bki_credit__mean,"95,492.7235",1225
396,imputed__first_salary_income,"91,510.3183",2870
5,incomeValue,"79,344.9697",2825
380,family__turnover__max,"75,128.8298",1119
11,hdb_bki_total_pil_max_limit,"55,773.9117",2531
295,log1p_abs__turn_cur_cr_avg_act_v2,"52,109.3494",292


Top-20 residual features


,feature,family,gain,split
63,meta__base_pred,base_meta,"26,303.3771",1555
75,proxy__salary_confident_gap_to_base,salary_consensus,"17,731.4910",1009
76,prob_gt_50k,cdf_probs,"16,046.0748",828
0,salary_6to12m_avg,direct_income,"13,439.5414",883
77,prob_gt_80k,cdf_probs,"13,133.2809",610
78,prob_gt_120k,cdf_probs,"10,782.9482",699
74,proxy__salary_confidence,salary_consensus,"9,790.7338",512
55,by_category__amount__sum__eoperation_type_name...,transactions_tail,"9,602.6769",899
54,by_category__amount__sum__eoperation_type_name...,transactions_tail,"9,092.4626",838
79,prob_gt_180k,cdf_probs,"8,603.0390",736


Residual importance by family


,family,gain,split,n_features,gain_share
2,cdf_probs,"71,225.5178",5067,8,0.1801
11,transactions_tail,"64,536.6798",6124,12,0.1632
0,base_meta,"34,703.4502",2030,10,0.0878
10,stable_signals,"34,581.9954",3325,9,0.0875
8,salary_consensus,"33,419.7168",2075,3,0.0845
1,cashflow_balances,"32,474.3484",3128,9,0.0821
5,direct_income,"31,933.4830",2555,8,0.0808
7,row_aggregates,"30,593.3077",3047,11,0.0774
3,credit_capacity,"30,570.2603",2997,7,0.0773
6,financial_ratios,"20,234.6101",1935,5,0.0512


## 11. Проверка итогового файла

Перед загрузкой проверяем число строк, порядок `id`, отсутствие пустых и бесконечных
значений, а также допустимый диапазон прогноза. Рядом сохраняется небольшой файл
`metadata.json` с параметрами, чтобы позднее можно было точно восстановить конфигурацию.

In [22]:
saved_submission = pd.read_csv(SUBMISSION_PATH, sep=";", decimal=",")
saved_prediction = saved_submission["predict"].to_numpy(dtype=float)

assert list(saved_submission.columns) == [ID_COL, "predict"]
assert len(saved_submission) == len(sample_submission) == 73_214
assert saved_submission[ID_COL].equals(sample_submission[ID_COL])
assert saved_submission[ID_COL].is_unique
assert np.isfinite(saved_prediction).all()
assert saved_prediction.min() >= TARGET_MIN
assert saved_prediction.max() <= TARGET_MAX

metadata = {
    "architecture": "one LightGBM base + one LightGBM residual",
    "base_feature_set": "engineered_v1",
    "base_features": len(base_features),
    "base_params": BASE_PARAMS,
    "base_final_iterations": base_final_iterations,
    "base_temporal_scores": base_scores.to_dict(orient="records"),
    "residual_variant": "compact_proxy",
    "residual_features": chosen_features,
    "residual_params": final_residual_params,
    "residual_choice": residual_choice,
    "salary_proxy_sources": SALARY_PROXY_SOURCES,
    "salary_proxy_features": SALARY_PROXY_FEATURES,
    "target_lookup_applied": False,
    "submission": str(SUBMISSION_PATH),
}
(OUTPUT_DIR / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

display(
    pd.Series(
        {
            "rows": len(saved_submission),
            "prediction_min": saved_prediction.min(),
            "prediction_median": np.median(saved_prediction),
            "prediction_mean": saved_prediction.mean(),
            "prediction_p99": np.quantile(saved_prediction, 0.99),
            "prediction_max": saved_prediction.max(),
            "applied_correction_median": np.median(chosen_strength * raw_test_correction),
            "applied_correction_p99": np.quantile(chosen_strength * raw_test_correction, 0.99),
        },
        name="value",
    ).to_frame()
)
print("QA PASSED")

,value
rows,"73,214.0000"
prediction_min,"20,000.0000"
prediction_median,"58,382.0502"
prediction_mean,"98,389.9897"
prediction_p99,"526,395.1285"
prediction_max,"1,290,249.1542"
applied_correction_median,612.2386
applied_correction_p99,"54,640.2107"


QA PASSED


## Итог

Модель состоит из двух понятных частей:

- основная модель оценивает общий уровень дохода;
- модель-поправка исправляет повторяющиеся ошибки основной модели;
- три зарплатных признака объединяют четыре источника без обучения третьей модели;
- коэффициент `0.50` защищает прогноз от слишком сильных изменений на редких клиентах.

Проверка дала `58 680` WMAE на May и `58 761` на June. Ориентир около 76k public
остаётся приблизительной оценкой, а не гарантированным результатом.

Финальный файл находится в `outputs/simple_76k_notebook/submission_simple_76k.csv`.